In [1]:
# Example for running a PyTorch repo in Kaggle
!pip install -r requirements.txt
# Or install specific missing dependencies:
!pip install timm==0.4.12 opencv-python scikit-learn

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
ERROR: Could not find a version that satisfies the requirement timm==0.4.12 (from versions: none)
ERROR: No matching distribution found for timm==0.4.12


In [1]:
import os, sys, subprocess
from pathlib import Path

def run(cmd): subprocess.run(cmd, shell=True, check=True)

run("pip install -q gdown opencv-python-headless")

import torch, cv2, numpy as np, pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR   = Path("/kaggle/working")
REPO_DIR   = WORK_DIR / "PraNet-V2"
BINARY_DIR = REPO_DIR / "binary_seg"
TEST_SIZE  = 352
# Change these paths to match your currently mounted Kaggle inputs
INPUT_DIR = Path("/kaggle/input")
HYPERKVASIR_ROOT = INPUT_DIR / "hyperkvasir_og"
PICCOLO_ROOT = INPUT_DIR / "piccolo_og"
POLYPDB_ROOT = INPUT_DIR / "polypdb_og"
LDPOLYP_ROOT = INPUT_DIR / "ldpolypvideo_og_dataset"
WEIGHT_IDS = {
    "RES-V2.pth": "1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF",                          # PraNet-V2 checkpoint
    "res2net50_v1b_26w_4s-3cf99910.pth": "1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z",   # backbone, needed by model __init__
}

# --- Constants required by FlexiblePolypDataset (were missing) ---
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)  # ImageNet mean
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)  # ImageNet std

In [3]:
def clone_repo():
    if not REPO_DIR.exists():
        run(f"git clone --depth 1 https://github.com/ai4colonoscopy/PraNet-V2.git {REPO_DIR}")
    else:
        print("Repo already present, skipping clone.")

clone_repo()

Cloning into '/kaggle/working/PraNet-V2'...


In [4]:
import shutil
from pathlib import Path

# Create directory expected by relative path '../models' from /kaggle/working
target_dir = Path("/kaggle/models")
target_dir.mkdir(parents=True, exist_ok=True)

# Copy/symlink the downloaded backbone weight
src_weight = BINARY_DIR / "models" / "res2net50_v1b_26w_4s-3cf99910.pth"
dst_weight = target_dir / "res2net50_v1b_26w_4s-3cf99910.pth"

if src_weight.exists() and not dst_weight.exists():
    shutil.copy(src_weight, dst_weight)

In [5]:
import gdown
import shutil

def download_weights():
    snapshot_dir = BINARY_DIR / "snapshots" / "PraNet-V2"
    models_dir = BINARY_DIR / "models"
    
    # Create the parent directory '../../models' that Res2Net_v1b explicitly looks for
    alt_models_dir = REPO_DIR.parent / "models"
    
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)
    alt_models_dir.mkdir(parents=True, exist_ok=True)

    targets = {
        "RES-V2.pth": snapshot_dir / "RES-V2.pth",
        "res2net50_v1b_26w_4s-3cf99910.pth": models_dir / "res2net50_v1b_26w_4s-3cf99910.pth",
    }

    for fname, dest in targets.items():
        if not (dest.exists() and dest.stat().st_size > 1_000_000):
            gdown.download(id=WEIGHT_IDS[fname], output=str(dest), quiet=False)

    # Symlink or copy backbone weight to the parent path expected by Res2Net_v1b
    alt_backbone = alt_models_dir / "res2net50_v1b_26w_4s-3cf99910.pth"
    if not alt_backbone.exists():
        shutil.copy(targets["res2net50_v1b_26w_4s-3cf99910.pth"], alt_backbone)
    kaggle_models = Path("/kaggle/models")
    kaggle_models.mkdir(parents=True, exist_ok=True)
    shutil.copy(targets["res2net50_v1b_26w_4s-3cf99910.pth"], kaggle_models / "res2net50_v1b_26w_4s-3cf99910.pth")
    return targets["RES-V2.pth"]

CKPT_PATH = download_weights()
# Add this inside cell 4 right after downloading:


Downloading...
From (original): https://drive.google.com/uc?id=1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF
From (redirected): https://drive.google.com/uc?id=1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF&confirm=t&uuid=b45439f6-3c54-4108-860e-e79974ca4b77
To: /kaggle/working/PraNet-V2/binary_seg/snapshots/PraNet-V2/RES-V2.pth
100%|██████████| 131M/131M [00:01<00:00, 95.0MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z
To: /kaggle/working/PraNet-V2/binary_seg/models/res2net50_v1b_26w_4s-3cf99910.pth
100%|██████████| 103M/103M [00:01<00:00, 83.7MB/s] 


In [6]:
import importlib
import sys
import os
import re
from pathlib import Path

def load_pranet_v2(ckpt_path: Path):
    if str(BINARY_DIR) not in sys.path:
        sys.path.insert(0, str(BINARY_DIR))
    if str(BINARY_DIR / "lib") not in sys.path:
        sys.path.insert(0, str(BINARY_DIR / "lib"))
    
    # Import PraNet model module directly
    try:
        from lib.pranet import PraNet_V2
        model = PraNet_V2(channel=32)
    except Exception:
        pranet_module = importlib.import_module("lib.pranet")
        # Fallback to PraNet if available
        model_cls = getattr(pranet_module, "PraNet", None) or getattr(pranet_module, "PraNet_V2")
        model = model_cls()

    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # Fix channel size mismatch on Conv and BatchNorm layers for binary segmentation
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.BatchNorm2d) and module.num_features == 3:
            weight_key = f"{name}.weight"
            if weight_key in state_dict and state_dict[weight_key].shape[0] == 1:
                parent_name, _, child_name = name.rpartition('.')
                parent = model if not parent_name else model.get_submodule(parent_name)
                setattr(parent, child_name, torch.nn.BatchNorm2d(1).to(DEVICE))
                
        elif isinstance(module, torch.nn.Conv2d) and module.out_channels == 3:
            weight_key = f"{name}.weight"
            if weight_key in state_dict and state_dict[weight_key].shape[0] == 1:
                new_conv = torch.nn.Conv2d(
                    in_channels=module.in_channels,
                    out_channels=1,
                    kernel_size=module.kernel_size,
                    stride=module.stride,
                    padding=module.padding,
                    bias=module.bias is not None
                )
                parent_name, _, child_name = name.rpartition('.')
                parent = model if not parent_name else model.get_submodule(parent_name)
                setattr(parent, child_name, new_conv)

    info = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded weights! Missing keys: {len(info.missing_keys)}, Unexpected: {len(info.unexpected_keys)}")
    
    return model.to(DEVICE).eval()

model = load_pranet_v2(CKPT_PATH)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/kaggle/working/PraNet-V2/binary_seg/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/PraNet-V2/binary_seg/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing 

Loaded weights! Missing keys: 0, Unexpected: 2


In [10]:
from torch.utils.data import DataLoader
import csv

def test_dataset(model, img_root, mask_root, output_csv="per_image_metrics.csv"):
    dataset = FlexiblePolypDataset(img_root, mask_root)
    if len(dataset) == 0:
        print("No paired data found. Skipping.")
        return
        
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    model.eval()
    
    results = []
    
    with torch.no_grad():
        for i, (images, masks, img_paths) in enumerate(loader):
            images = images.to(DEVICE)
            masks = masks.numpy().squeeze()
            
            # PraNet usually returns a tuple of 4 multiscale predictions. 
            # The first or last element is the final prediction map.
            preds = model(images)
            pred = preds[0] if isinstance(preds, (list, tuple)) else preds
            
            # Normalize and threshold prediction
            pred = F.sigmoid(pred)
            pred_np = (pred > 0.5).float().cpu().numpy().squeeze()
            
            dice, iou = dice_iou(pred_np, masks)
            
            results.append({
                "Image_Path": img_paths[0],
                "Dice": dice,
                "IoU": iou
            })
            
            if i % 100 == 0:
                print(f"Processed {i}/{len(loader)} images...")

    # Save to CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\n--- Metrics for {Path(img_root).name} ---")
    print(f"Mean Dice: {df['Dice'].mean():.4f}")
    print(f"Mean IoU:  {df['IoU'].mean():.4f}")
    print(f"Saved full results to {output_csv}\n")

# Run testing (Example for Piccolo dataset)
# Ensure you point these to the exact folders containing the images and masks
piccolo_img_dir = PICCOLO_ROOT / "Piccolo dataset-release0.1" / "images" # adjust subfolder if necessary
piccolo_mask_dir = PICCOLO_ROOT / "Piccolo dataset-release0.1" / "masks"

test_dataset(model, piccolo_img_dir, piccolo_mask_dir, output_csv="/kaggle/working/piccolo_metrics.csv")

Found 0 image-mask pairs in images
No paired data found. Skipping.


In [13]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
import torch.nn.functional as F

# --- Metrics Helper ---
def dice_iou(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    dice = (2.0 * intersection) / (pred.sum() + gt.sum() + 1e-8)
    iou = intersection / (union + 1e-8)
    return float(dice), float(iou)

# --- Fixed Dataset Class ---
# --- Fixed Dataset Class ---
class FlexiblePolypDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_exts=IMG_EXT):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        
        # Find all images normally
        raw_img_paths = [p for p in self.img_dir.rglob("*") if p.suffix.lower() in img_exts]
        
        # Find masks, explicitly allowing .txt files
        raw_mask_paths = [p for p in self.mask_dir.rglob("*") if p.suffix.lower() in img_exts or p.suffix.lower() == '.txt']
        
        # Include the parent directory (e.g., '101') in the key to prevent filename collisions
        mask_dict = {m.stem: m for m in raw_mask_paths}       
        self.pairs = []
        for img_path in raw_img_paths:
            key = img_path.stem
            if key in mask_dict:
                self.pairs.append((img_path, mask_dict[key]))
                
        print(f"Found {len(self.pairs)} image-mask pairs in {self.img_dir.name}")
        print(raw_img_paths[:3])
        print(raw_mask_paths[:3])
    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        
        # Read image to get original dimensions before resizing
        image = cv2.imread(str(img_path))
        orig_h, orig_w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (TEST_SIZE, TEST_SIZE))
        image = (image / 255.0 - MEAN) / STD
        image = image.transpose(2, 0, 1).astype(np.float32)
        
        # Handle text-based YOLO bounding boxes
        if mask_path.suffix.lower() == '.txt':
            mask = np.zeros((orig_h, orig_w), dtype=np.float32)
            # Only process if the file has enough bytes to contain coordinates
            if mask_path.stat().st_size > 5: 
                with open(mask_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        # Assuming standard YOLO format: class x_center y_center width height
                        if len(parts) >= 5:
                            _, x_c, y_c, w, h = map(float, parts[:5])
                            
                            # Convert relative center coordinates to absolute pixel boundaries
                            x1 = int(max(0, (x_c - w / 2) * orig_w))
                            y1 = int(max(0, (y_c - h / 2) * orig_h))
                            x2 = int(min(orig_w, (x_c + w / 2) * orig_w))
                            y2 = int(min(orig_h, (y_c + h / 2) * orig_h))
                            
                            # Draw rectangular mask
                            mask[y1:y2, x1:x2] = 1.0
                            
            # Resize the generated pseudo-mask
            mask = cv2.resize(mask, (TEST_SIZE, TEST_SIZE), interpolation=cv2.INTER_NEAREST)
            
        # Handle normal image masks if mixed formats exist
        else:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (TEST_SIZE, TEST_SIZE), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
            
        mask = np.expand_dims(mask, axis=0)
        return torch.tensor(image), torch.tensor(mask), str(img_path)

# --- Execution Call ---
# Define precise paths based on your dataset summary
ldpolyp_root = Path("/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test")
ldpolyp_img_dir = ldpolyp_root / "Images" # Presumed based on standard structuring alongside Annotations
ldpolyp_mask_dir = ldpolyp_root / "Annotations"

# Run the test_dataset function
test_dataset(model, ldpolyp_img_dir, ldpolyp_mask_dir, output_csv="/kaggle/working/ldpolyp_per_image_metrics.csv")
# --- Per-Image Evaluation Function ---
@torch.no_grad()
def evaluate_dataset_per_image(model, dataset: Dataset, dataset_name: str, threshold: float = 0.5):
    records = []
    
    if len(dataset) == 0:
        print(f"Skipping {dataset_name}: No image pairs found.")
        return records

    print(f"Evaluating {len(dataset)} images for {dataset_name}...")
    for idx in range(len(dataset)):
        tensor, gt, (orig_h, orig_w), img_id = dataset[idx]
        tensor = tensor.unsqueeze(0).to(DEVICE)
        
        # PraNet forward pass (returns lateral maps; lateral_map_2 is index -1)
        outputs = model(tensor)
        res = outputs[-1] if isinstance(outputs, (tuple, list)) else outputs

        # Upsample to original mask resolution
        res = F.interpolate(res, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
        pred_prob = torch.sigmoid(res).squeeze().cpu().numpy()
        pred_mask = (pred_prob > threshold).astype(np.float32)
        gt_np = gt.numpy()

        dice, iou = dice_iou(pred_mask, gt_np)

        records.append({
            "dataset": dataset_name,
            "image_id": img_id,
            "dice": round(dice, 4),
            "iou": round(iou, 4)
        })

    return records

# --- Run Evaluation Across Datasets ---
all_image_metrics = []

# Assuming your datasets are directly under /kaggle/input/ or a specific user directory
# Adjust the base prefix if yours is nested (e.g., /kaggle/input/datasets/varshasrao/...)

# Change these paths to match your currently mounted Kaggle inputs
INPUT_DIR = Path("/kaggle/input")
HYPERKVASIR_ROOT = INPUT_DIR / "hyperkvasir_og"
PICCOLO_ROOT = INPUT_DIR / "piccolo_og"
POLYPDB_ROOT = INPUT_DIR / "polypdb_og"
LDPOLYP_ROOT = INPUT_DIR / "ldpolypvideo_og_dataset"
datasets_to_eval = {
    "Piccolo": PICCOLO_ROOT,
    "PolypDB": POLYPDB_ROOT,
    "LDPolyp": LDPOLYP_ROOT
}
for name, path in datasets_to_eval.items():
    # Fallback to the nested path you used previously if the direct path fails
    if not path.exists():
        fallback_path = Path(str(path).replace("/kaggle/input/", "/kaggle/input/datasets/varshasrao/"))
        if fallback_path.exists():
            path = fallback_path
        
    if not path.exists():
        print(f"⚠️ Skipping {name}: Directory not found at {path}")
        continue
    
    dataset = FlexiblePolypDataset(path)
    metrics = evaluate_dataset_per_image(model, dataset, name)
    all_image_metrics.extend(metrics)

# --- Save and Inspect Per-Image Results ---
df_per_image = pd.DataFrame(all_image_metrics)

csv_save_path = WORK_DIR / "per_image_metrics_all.csv"
df_per_image.to_csv(csv_save_path, index=False)
print(f"\n✅ Per-image metrics for all datasets saved to: {csv_save_path}")

# Summary by dataset
print("\nDataset Summary Averages:")
print(df_per_image.groupby("dataset")[["dice", "iou"]].mean())

Found 15397 image-mask pairs in Images
[PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Images/135/0106.jpg'), PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Images/135/0375.jpg'), PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Images/135/0285.jpg')]
[PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Annotations/135/0389.txt'), PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Annotations/135/0078.txt'), PosixPath('/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Annotations/135/0333.txt')]
Processed 0/15397 images...
Proc

KeyError: 'dataset'